# Imports

In [29]:
import requests
import pandas as pd
from io import StringIO
from src.data_loader import load_reservoir_data
from src.mongo_loader import load_mongo_data
from src.weather_loader import download_weather
import os
from pathlib import Path
import sys


# Get exchange rates

In [13]:
def get_exchange_rates(start_date, end_date):
    url = "https://data.norges-bank.no/api/data/EXR/B.EUR+USD.NOK.SP"

    params = {
        "startPeriod": start_date,
        "endPeriod": end_date,
        "format": "csv"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text), sep=";")

    #Extracting relevant columns

    df = df[["TIME_PERIOD", "BASE_CUR", "OBS_VALUE"]].copy()

    df.columns = ["date", "currency", "exchange_rate"]

    df["date"] = pd.to_datetime(df["date"])
    df["exchange_rate"] = pd.to_numeric(df["exchange_rate"])

    return df

In [14]:
exchange_rates = get_exchange_rates("2026-01-01", "2026-01-31")

display(exchange_rates.head())

,date,currency,exchange_rate
0,2026-01-02,USD,10.0661
1,2026-01-05,USD,10.0990
2,2026-01-06,USD,10.0184
3,2026-01-07,USD,10.0638
4,2026-01-08,USD,10.1015


# Checking timezones


In [16]:
df = load_reservoir_data()
print("NVE:")
print(df["date"].dtype)
print(df["date"].head())

print("Norges Bank:")
print(exchange_rates["date"].dtype)
print(exchange_rates["date"].head())

NVE:
datetime64[us]
14903   1995-01-08
9951    1995-01-08
7718    1995-01-08
11268   1995-01-08
7118    1995-01-08
Name: date, dtype: datetime64[us]
Norges Bank:
datetime64[us]
0   2026-01-02
1   2026-01-05
2   2026-01-06
3   2026-01-07
4   2026-01-08
Name: date, dtype: datetime64[us]


In [30]:
# Find the project root
project_root = Path.cwd().parent

# Change working directory to project root
os.chdir(project_root)

# Make src importable
sys.path.insert(0, str(project_root))

print("Current directory:", Path.cwd())
print("Secrets file exists:", (project_root / ".streamlit" / "secrets.toml").exists())

Current directory: c:\Users\magnu\OneDrive\Dokumenter\IND320\magloevIND320
Secrets file exists: True


In [33]:
weather_df = download_weather(
    "NO 3",
    "2024-01-01",
    "2024-12-31"
)

entsoe_df = load_mongo_data()

Loading saved batch 0
Loading saved batch 100
Loading saved batch 200
Loading saved batch 300
Loading saved batch 400
Loading saved batch 500
Loading saved batch 600
Loading saved batch 700
Loading saved batch 800
Loading saved batch 900


In [40]:
print("Open-meteo:")
print(weather_df["date"].dtype)
print(weather_df["date"].head())

print("ENTSO-E:")
print(entsoe_df["timestamp"].dtype)
print(entsoe_df["timestamp"].head())

Open-meteo:
datetime64[us, UTC]
0   2024-01-01 00:00:00+00:00
1   2024-01-01 01:00:00+00:00
2   2024-01-01 02:00:00+00:00
3   2024-01-01 03:00:00+00:00
4   2024-01-01 04:00:00+00:00
Name: date, dtype: datetime64[us, UTC]
ENTSO-E:
datetime64[us]
0   2024-10-09 02:00:00
1   2024-10-10 11:00:00
2   2024-10-14 17:00:00
3   2024-10-15 08:00:00
4   2024-10-15 10:00:00
Name: timestamp, dtype: datetime64[us]


# AI Usage

# Log